# LLMOps · RAGOps 아키텍처 및 평가 개론

- LLMOps의 3대 핵심 축인 **관측(Observability), 평가(Evaluation), 운영(Ops)** 의 전체 구조를 이해한다.
- **Trace, Span, Score**의 계층 구조와 비용/지연 시간 귀속 메커니즘을 파악한다.
- 다축 평가(Multi-axis Eval)와 **회귀 방지 배포 게이트(Deployment Gate)** 의 설계 원칙을 정립한다.
- 핵심 관측/평가 5대 질문에 대해 **동작 원리를 예측하고 검증 기준을 수립**한다.

---

## 0. 실행 준비

이 노트북은 `.env`의 `OPENAI_API_KEY`를 필수로 사용하며, 선택적으로 `LANGFUSE_*` 키를 지원합니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY 가 없습니다. 프로젝트 루트에 .env 를 만들고 키를 넣으세요."
)
print("기본 환경 준비 완료")

### 🔌 관측 도구 연동 (Langfuse — 부가 기능)

Langfuse 키가 설정되어 있으면 원격 대시보드에 트레이스를 전송하고, 없더라도 로컬 콜백(`Spy`)을 통해 모든 스팬과 토큰을 완벽히 관찰할 수 있도록 구성합니다.

In [ ]:
# Langfuse 연결 시도 — 연결 실패 시에도 로컬 실습이 중단되지 않도록 안전하게 처리 (Graceful Fallback)
TRACE_CONFIG: dict = {}

try:
    # Langfuse Python SDK v3/v4 표준 import 경로
    from langfuse.langchain import CallbackHandler
    if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
        handler = CallbackHandler()
        TRACE_CONFIG = {"callbacks": [handler]}
        print("✅ [Langfuse] 원격 트레이싱 활성화 완료")
    else:
        print("ℹ️ [Langfuse] 키 미설정 — 로컬 콜백(Spy) 모드로 동작합니다.")
except Exception as e:
    print(f"⚠️ [Langfuse] 초기화 생략 ({e}) — 로컬 콜백(Spy) 모드로 계속합니다.")

> ### 학습 방향
>
>  **"우리가 만든 시스템을 어떻게 관측하고, 평가하며, 안전하게 배포·운영할 것인가"** 에 대한 엔터프라이즈 기준을 세우는 블록입니다.
>
> - **관측(Observability)**: 파이프라인 내부의 지연과 비용을 어디에 귀속시킬 것인가
> - **평가(Evaluation)**: 평균의 함정에 빠지지 않고 문항별 회귀를 어떻게 감지할 것인가
> - **운영(Operations)**: 배포 게이트와 재시도 정책을 어떻게 결정론적으로 제어할 것인가

---

## 1. 관측(Observability)의 3대 핵심 요소 — Trace, Span, Score

| 구성 요소 | 정의 | 기록 시점 및 내용 |
|---|---|---|
| **Trace (트레이스)** | 단일 사용자 요청의 전체 실행 수명주기 | 요청 시작 시점 생성, 고유 ID 및 전체 성공/실패 상태 |
| **Span (스팬)** | 트레이스 내부의 세부 실행 단위 (LLM 호출, 체인, 도구, 검색) | 각 단계 시작/종료 시점, 소요 시간, 입출력 데이터 |
| **Score (스코어)** | 트레이스 또는 스팬 단위로 매겨지는 정량적 평가값 | 실행 완료 후 또는 비동기 평가기(정확도, 사용자 피드백 등) |

### 🔍 실습: Langfuse 원격 트레이싱 단계별 연동 및 스팬 관측

LangChain 파이프라인에 Langfuse 콜백(`CallbackHandler`)을 연결하여 **① 요청 단위 트레이스(Trace) 생성**, **② 메타데이터 및 태그 주입**, **③ 세부 스팬(Span) 및 토큰 소비량 추적**을 단계별로 실습합니다.

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# [1단계] LLM 및 체인 구성
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 사내 LLMOps 및 Observability 전문가입니다. 핵심 위주로 명확히 답하세요."),
    ("human", "{question}")
])
chain = prompt | llm

In [ ]:
# [2단계] 트레이스 실행 및 메타데이터 주입
QUESTION = "LLMOps 관측 시스템에서 Trace와 Span의 핵심 차이는 무엇인가요?"

# TRACE_CONFIG에 등록된 Langfuse 핸들러 또는 로컬 Spy 핸들러와 함께 메타데이터 전달
invoke_config = {
    **TRACE_CONFIG,
    "metadata": {
        "langfuse_user_id": "engineer_01",
        "langfuse_session_id": "session_p1_04_intro",
        "langfuse_tags": ["part1", "llmops-intro", "edu-demo"],
        "environment": "development",
    },
}

print(f"질문: \"{QUESTION}\"")
response = chain.invoke({"question": QUESTION}, config=invoke_config)

print("\n[체인 실행 완료]")
print(response.content)


In [ ]:
# [3단계] Langfuse 트레이스 확인 안내
if TRACE_CONFIG.get("callbacks"):
    try:
        from langfuse import Langfuse
        Langfuse().flush()  # 비동기 전송 버퍼 플러시
        print("\n" + "=" * 64)
        print("✅ Langfuse 대시보드로 트레이스 데이터가 성공적으로 전송되었습니다!")
        print("👉 Langfuse 대시보드(Traces 메뉴)에서 위 session_id와 세부 Span 계층을 확인하세요.")
        print("=" * 64)
    except Exception as e:
        print(f"\n(트레이스 플러시 안내: {e})")
else:
    print("\nℹ️ 로컬 모드 실행: Langfuse API 키 설정 시 대시보드에서 실시간 분산 트레이스를 확인할 수 있습니다.")


## 🔮 사전 질문 ① — 트레이스 성공률과 비즈니스 실패율

**"노드 내부에서 예외를 `try-except`로 잡아서 처리하면, 대시보드 에러율은 몇 %로 찍힐까요?"**

<details>
<summary>생각해보기</summary>

> **질문의 핵심:** 노드 내부에서 예외를 잡아 대체 텍스트를 반환하면 애플리케이션 프로세스는 정상 종료되므로, 관측 도구는 이를 '성공(에러율 0%)'으로 기록합니다. 즉, 비즈니스 실패율(예: 33%)과 대시보드 에러율 간에 심각한 괴리가 발생합니다. 

</details>

---

## 3. 메타데이터(Metadata)와 태그(Tags)의 역할

- **Tags & Metadata (입력 시점에 부여)**: 세션 ID, 사용자 부서, 환경(`prod`/`staging`), 프롬프트 버전 등 요청 시점에 이미 알고 있는 컨텍스트.
- **Scores & Feedback (실행 완료 후 부여)**: 실제 답변의 정확도, 기권 여부, 응답 지연 시간, 사용자 만족도 등 결과 관찰 후 계산되는 지표.

In [ ]:
import os
from langfuse import Langfuse

# [실습] Langfuse Python SDK를 활용한 태그(Tags) 기반 원격 트레이스 검색
if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
    langfuse = Langfuse()
    
    # 1. 방금 주입한 태그("llmops-intro")로 트레이스 목록 쿼리
    TARGET_TAG = "llmops-intro"
    print(f"🔍 태그 [\x27{TARGET_TAG}\x27] 기준 트레이스 검색 중...\n")
    
    try:
        traces = langfuse.api.trace.list(tags=TARGET_TAG, limit=3)
        print(f"총 {len(traces.data)}개의 트레이스가 검색되었습니다:\n")
        
        for i, t in enumerate(traces.data, 1):
            print(f"[{i}] Trace ID : {t.id}")
            print(f"    • Name    : {t.name}")
            print(f"    • User/Session : {t.user_id} / {t.session_id}")
            print(f"    • Tags    : {t.tags}")
            print(f"    • Latency : {t.latency:.2f}s")
            print(f"    • Input   : {str(t.input)[:60]}...")
            print("-" * 54)
    except Exception as e:
        print(f"트레이스 조회 중 오류: {e}")
else:
    print("ℹ️ Langfuse API 키가 설정되지 않았습니다. .env에 키를 설정하면 SDK 원격 쿼리를 테스트할 수 있습니다.")


---

## 4. 비용과 지연 시간의 분리 — 스팬별 귀속 원리

에이전트 워크플로우를 관측할 때 가장 중요한 원칙 중 하나는 **비용과 지연 시간의 귀속 지점이 다르다**는 사실입니다.

## 🔮 사전 질문 ② — 노드 스팬과 LLM 스팬의 비용 귀속

**"LangGraph 노드 스팬과 그 안에서 호출된 LLM 스팬 중, 실제 API 토큰 비용은 어디에 청구될까요?"**

<details>
<summary>생각해보기</summary>

> **질문의 핵심:** 토큰 비용은 오직 **LLM 스팬(실제 API를 호출한 줄)** 에만 귀속됩니다. 노드나 체인 스팬은 비용이 0원이며, 오직 **지연 시간(Latency)과 데이터 흐름을 추적하기 위한 컨테이너** 역할을 합니다.

</details>

In [ ]:
import time
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class Spy(BaseCallbackHandler):
    """스팬 계층 구조, 소요 시간(Latency), 토큰 소비를 로컬에서 관측하는 핸들러"""
    def __init__(self):
        self.rows = []
        self._starts = {}

    def on_chain_start(self, serialized, inputs, *, run_id, **kwargs):
        self._starts[run_id] = time.perf_counter()
        name = (serialized or {}).get("id", ["Chain"])[-1]
        self.rows.append({"run_id": run_id, "kind": "Chain", "name": name, "in_sz": len(str(inputs)), "tok": None, "dur": None})

    def on_chain_end(self, outputs, *, run_id, **kwargs):
        dur = time.perf_counter() - self._starts.pop(run_id, time.perf_counter())
        for r in self.rows:
            if r["run_id"] == run_id:
                r["dur"] = dur

    def on_llm_start(self, serialized, prompts, *, run_id, **kwargs):
        self._starts[run_id] = time.perf_counter()
        name = (serialized or {}).get("name", "LLM")
        self.rows.append({"run_id": run_id, "kind": "LLM", "name": name, "in_sz": sum(len(p) for p in prompts), "tok": None, "dur": None})

    def on_llm_end(self, response, *, run_id, **kwargs):
        dur = time.perf_counter() - self._starts.pop(run_id, time.perf_counter())
        usage = (response.llm_output or {}).get("token_usage", {})
        tot = usage.get("total_tokens")
        for r in self.rows:
            if r["run_id"] == run_id:
                r["tok"] = tot
                r["dur"] = dur


# 테스트 체인 구성 및 실행
prompt = ChatPromptTemplate.from_messages([("human", "{q}")])
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
chain = prompt | llm

spy = Spy()
cfg = {"callbacks": [spy, *TRACE_CONFIG.get("callbacks", [])]}
chain.invoke({"q": "LLMOps 의 핵심 3요소는?"}, config=cfg)

hdr_kind, hdr_name, hdr_in, hdr_dur, hdr_tok = "스팬 유형", "스팬 이름", "입력 크기", "소요 시간(Latency)", "토큰 비용"
print(f"{hdr_kind:<10} {hdr_name:<24} {hdr_in:>10} {hdr_dur:>18} {hdr_tok:>14}")
print("-" * 80)
for r in spy.rows:
    tok_val = r["tok"]
    dur_val = r["dur"]
    tok_str = f"{tok_val} 토큰" if tok_val is not None else "- (비용 0)"
    dur_str = f"{dur_val * 1000:.1f}ms" if dur_val is not None else "-"
    k, n, s = r["kind"], r["name"], r["in_sz"]
    print(f"{k:<10} {n:<24} {s:>8}자 {dur_str:>18} {tok_str:>14}")


> ### ▶ 함께 실행 — D24
>
> **체인을 1회 실행하고 로컬 콜백(Spy)에 수집된 스팬 구조를 확인합니다.**
>
> 🔍 **확인 사항** — 전체 스팬 수와 토큰이 부과된 스팬 수를 확인하세요.
>- `Chain` 스팬: 토큰 비용 `- (비용 0)`
>- `LLM` 스팬: 실제 소비된 토큰 수량 기록
>
> ⚠️ **참고**: 실행 결과의 수치는 실행 환경이나 시점에 따라 다를 수 있습니다. 이는 오류가 아니며, 생성 모델의 통계적 변동성을 직접 확인하는 실습입니다.

---

### 📊 결과 해석

| 결과 상황 | 해설 및 원인 분석 |
|---|---|
| 예상대로 출력됨 (LLM 스팬에만 토큰 기록) | 프롬프트 체인, 라우터, 상태 노드는 오케스트레이션 역할을 수행하므로 자체 토큰 비용이 0원입니다. 비용 청구는 오직 LLM API를 직접 호출한 스팬에만 집중됩니다. |
| 스팬 수가 다르게 표시되는 경우 | LangChain 내부 버전이나 프롬프트 직렬화 방식에 따라 중간 체인 스팬이 세분화될 수 있습니다. 중요한 점은 LLM 스팬의 개수와 토큰 수입니다. |

> 🎯 **핵심 아키텍처 원칙: 비용(Cost)과 지연 시간(Latency)의 귀속 지점 분리**
>
> 1. **비용(Cost)의 귀속 지점 ➔ `LLM / Generation` 스팬**
>    - 실제 토큰(Input/Output Tokens)이 소모되고 과금되는 곳은 오직 LLM API를 직접 호출한 스팬뿐입니다.
>    - 프롬프트 체인, 상태 그래프 노드, 라우터 자체의 토큰 비용은 **항상 0원**입니다.
>
> 2. **지연 시간(Latency) 및 흐름의 귀속 지점 ➔ `Chain / Node` 스팬**
>    - 노드와 체인 스팬은 DB 쿼리, 파싱, 직렬화, 네트워크 I/O 등 **전체 처리 시간의 병목 구간**을 진단하기 위한 실행 컨테이너입니다.

| 스팬 유형 | 주 목적 | 모니터링 핵심 지표 | 최적화 대상 |
|---|---|---|---|
| **Chain / Node 스팬** | 파이프라인 흐름 & 병목 진단 | **소요 시간 (Latency, 초)** | I/O 병렬화, 불필요한 직렬화/파싱 오버헤드 제거 |
| **LLM (Generation) 스팬** | API 실행 & 토큰 과금 추적 | **입력/출력 토큰 수 & 비용 ($)** | 프롬프트 군더더기 제거, 모델 티어 다운그레이드 |

---

## 5. 평가(Evaluation) 체계 구축 — 골든셋과 다축 지표

### 📐 다축 평가(Multi-axis Evaluation)의 필요성

단일 지표(예: ROUGE 스코어)만으로는 복합적인 에이전트 시스템을 평가할 수 없습니다. 실무에서는 다음과 같은 **다축 지표**를 동시에 측정합니다.

1. **검색 품질 (Retrieval)**: Context Precision(정답 포함률), Context Recall(재현율)
2. **생성 품질 (Generation)**: Faithfulness(환각 없는 충실도), Answer Relevance(답변 적합도)
3. **거버넌스 및 안전성 (Safety)**: 기권율(Abstention Rate), 보안 정책 준수율

---

## 🔮 사전 질문 ③ — 지표 간 상충 관계(Trade-off)

**"프롬프트를 수정하여 환각(Faithfulness)을 개선했을 때, 답변 적합성(Relevance)이나 기권율은 어떻게 변할까요?"**

<details>
<summary>생각해보기</summary>

> **질문의 핵심:** 지표들은 독립적이지 않고 서로 상충(Trade-off)합니다. 엄격한 환각 방지 지침을 추가하면 충실도는 올라가지만, 조금만 불확실해도 답변을 포기하여 기권율이 급증할 수 있습니다. 따라서 복수의 지표를 동시에 관찰해야 합니다.

</details>

---

## 6. 배포 게이트(Deployment Gate)와 회귀 방지

### 💡 게이트 설계 원칙 — 실측 기반 기준선과 래칫(Ratchet) 원리

배포 게이트의 임계값(Threshold)을 감으로 정하면 시스템이 첫날부터 동작하지 않거나 장식으로 전락합니다.

```text
① 현재 기준선(Baseline)을 정밀 실측한다.
② 실측치 바로 아래에 안전 하한선을 설정한다.
③ 시스템이 개선되면 하한선을 단계적으로 상향(Ratchet)한다.
```

> 🔒 **래칫 원칙**: 한 번 올린 배포 하한선은 특별한 사유 없이 임의로 낮추지 않습니다. 하한선을 통과하지 못할 때는 임계값을 내리는 것이 아니라 배포를 중단하고 롤백해야 합니다.

---

## 🔮 사전 질문 ④ — 평균 상승과 배포 결정

**"전체 평균 점수가 0.70에서 0.74로 올랐습니다. 이 모델을 즉시 프로덕션에 배포해도 될까요?"**

<details>
<summary><strong>평균의 함정 및 문항별 회귀(Regression) 예시 펼치기</strong></summary>

전체 테스트셋의 평균 점수가 상승했다고 해서 시스템이 개선되었다고 단정할 수 없습니다.

```text
버전 A 전체 평균: 0.700
버전 B 전체 평균: 0.740  (+0.040 상승, 개선된 것처럼 보임)
```

그러나 문항별 세부 결과를 비교하면 심각한 **회귀(Regression)** 가 숨어 있을 수 있습니다.

```text
Q07 (결제 규정): 통과 → 실패 ❌  [회귀 발생!]
Q13 (출장비 규정): 통과 → 실패 ❌  [회귀 발생!]
Q02 (일반 문의): 실패 → 통과 ✅
Q05 (인사 규정): 실패 → 통과 ✅
```

평균은 상승했지만 기존에 정상 작동하던 핵심 비즈니스 질문(Q07, Q13)에서 치명적인 오류가 발생했습니다.

> 🎯 **배포 평가 2대 규칙**:
> 1. **고유 ID(`case_id`) 기반 조인**: 단순 순서로 비교하지 말고 고유 식별자를 기준으로 1:1 diff를 수행합니다.
> 2. **`None`(측정 불가)과 `0점`의 분리**: 도구 미호출 등으로 점수 측정이 불가능한 케이스를 0점으로 왜곡하지 않고 명확히 분리합니다.

</details>

---

## 6. LLMOps 성숙도 진단 체크리스트

| 성숙도 단계 | 주요 특성 | 달성 기준 |
|---|---|---|
| **Level 0 (기초)** | 로깅 부재, 감에 의한 수정 | 에러 로그만 콘솔에 출력 |
| **Level 1 (관측)** | Trace / Span 추적 | Langfuse 등 트레이싱 도구로 입출력/비용 시각화 |
| **Level 2 (평가)** | 골든셋 기반 다축 평가 | 회귀 감지 문항(`trap` 포함) 및 정량 지표 측정 |
| **Level 3 (운영/게이트)** | CI/CD 배포 게이트 자동화 | 하한선 래칫 및 회귀 발생 시 자동 배포 차단 |

| 흔한 오해 | 실제 아키텍처 원리 |
|---|---|
| 관측·평가·운영은 프로덕션 배포 직전에 붙이는 부가 기능이다 | **만드는 것과 신뢰할 수 있게 검증하는 것은 완전히 별개의 작업**이며, 무엇을 계측하고 기록할지는 개발 초기 단계부터 아키텍처에 설계되어야 합니다. |

---

## 8. Part 2 로드맵 — 실무 에이전트 구축 여정

Part 1에서 정립한 핵심 개념들을 바탕으로, Part 2에서는 실제 엔터프라이즈 환경에서 작동하는 고신뢰성 에이전트 시스템을 단계별로 구축합니다.

| 블록 | 핵심 주제 | 실전 구축 내용 | 아키텍처 연계 원리 |
|:---:|---|---|---|
| **2-1** | **Tool 연동 & API 통합** | `@tool` 기반 사내 DB/API 연동 및 **4단계 에러 처리 계약** 구현 | 도구 스키마 정밀화 & 장애 격리 |
| **2-2** | **ReAct Agent 구현 & 제어** | ReAct 루프 직접 구현, **실행 상한(Limit) 및 토큰 예산** 통제 | 자율성 통제 & 무한 루프 폭주 방어 |
| **2-3** | **LangGraph 기반 상태 관리** | 상태 그래프 설계, **State · Reducer · Checkpointer** 영속화 | 결정론적 경로 제어 & 실행 복원력 |
| **2-4** | **분기 · 반복 · 병렬 워크플로우** | 조건부 동적 라우팅, **게이트 재시도 및 `Send` 기반 병렬 맵-리듀스** | 실행 지연(Latency) 단축 & 처리량 극대화 |
| **2-5** | **엔터프라이즈 자동화 프로젝트** | 엔드투엔드 고객 문의 처리 시스템 및 **인간 승인 게이트(HITL)** | 위험 업무 거버넌스 & 안전한 자동화 |
| **2-6** | **Observability & LLMOps** | **Langfuse 분산 트레이싱**, 결정론적 회귀 평가 & 배포 게이트 | 품질 모니터링 & 프로덕션 신뢰성 확보 |

---

### 🌉 Part 1 ➔ Part 2 아키텍처 핵심 연결고리

1. **LLM은 상태가 없다 (Stateless)**
   ➔ Agent는 대화 이력을 누적하므로, **도구 관찰 데이터(Observation)의 크기 통제와 미들웨어 기반 요약 압축**이 비용 절감의 핵심입니다.

2. **모델은 도구를 직접 실행하지 않는다 (Execution Boundary)**
   ➔ 실행 권한, 타임아웃, 예외 복구 지점은 **모두 우리 애플리케이션 코드의 통제 영역**에 있습니다.

3. **자율성은 온/오프가 아닌 연속된 스펙트럼이다 (Spectrum of Autonomy)**
   ➔ 비즈니스 위험도와 업무 정형성에 맞춰 **LangGraph의 상태 그래프로 최적의 제어 흐름(Control Flow)** 을 설계합니다.

---

## 정리

- **Trace · Span · Score**: 요청 시작 시점에는 Metadata/Tags를, 완료 후에는 Scores/Evaluation을 매핑합니다.
- **비용 귀속 원칙**: 비용은 LLM 스팬에만 부과되며, 노드/체인 스팬은 지연 시간(Latency) 및 컨텍스트 추적용입니다.
- **다축 평가 체계**: 단일 지표에 의존하지 않고 충실도, 관련성, 기권율을 종합적으로 측정합니다.
- **평균의 함정 극복**: `case_id` 기반 1:1 비교를 통해 평균 점수 뒤에 숨은 개별 문항의 회귀(Regression)를 감지합니다.
- **배포 게이트 래칫**: 실측치 기반으로 안전 하한선을 설정하고, 시스템 개선 시 단계적으로 상향하여 품질을 보증합니다.
- **Warn, Don't Crash**: 관측/보조 도구 장애가 비즈니스 서비스의 중단으로 이어지지 않도록 예외를 격리합니다.